<a href="https://colab.research.google.com/github/tabang205/Khai_Pha_Du_Lieu/blob/main/Scam_detection_training_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U pip -q
!pip install -U torchao peft transformers datasets accelerate pyarrow underthesea matplotlib -q

In [ ]:
import os, json, time, zipfile, math
import numpy as np
from pathlib import Path
from datetime import datetime
from itertools import cycle

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

from datasets import load_dataset, Dataset, DatasetDict, load_from_disk, Features, Value, ClassLabel
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
import shutil
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import f1_score, precision_score, recall_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
CONFIG = {
    # --- Paths ---
    'zip_path'    : '/content/drive/MyDrive/Học sâu 252AI31001/Data_Preprocessing.zip',
    'extract_dir' : '/content/parquet_files',
    'cache_dir'   : '/content/hf_cache',
    'dataset_path': '/kaggle/input/datasets/bangta205/train-val-test',
    'output_dir'  : '/kaggle/working/',

    # --- Model ---
    'model_name'  : 'vinai/phobert-base',
    'max_length'  : 128,
    'lora_r'      : 16,
    'lora_alpha'  : 32,
    'lora_dropout': 0.1,

    'batch_size'         : 64,
    'grad_accum_steps'   : 2,
    'learning_rate'      : 3e-5,
    'total_virtual_epochs': 20,
    'steps_per_vepoch'   : 5000,
    'warmup_steps'       : 1_000,
    'grad_clip'          : 1.0,

    # --- Focal Loss ---
    'focal_gamma' : 2.0,
    'focal_alpha' : 0.25,

    # --- Checkpoint ---
    'early_stop_patience': 5,

    # --- Data ---
    'val_size'  : 0.05,
    'test_size' : 0.05,
    'seed'      : 42,
    'num_proc'  : 4,
}

total_steps = CONFIG['total_virtual_epochs'] * CONFIG['steps_per_vepoch']

for p in ['output_dir', 'extract_dir', 'cache_dir']:
    Path(CONFIG[p]).mkdir(parents=True, exist_ok=True)

CKPT_PATH       = os.path.join(CONFIG['output_dir'], 'checkpoint.pt')
BEST_MODEL_PATH = os.path.join(CONFIG['output_dir'], 'best_model.pt')
LOG_PATH        = os.path.join(CONFIG['output_dir'], 'training_log.json')

print(f'Config OK | Device: {device}')
print(f'Virtual epochs: {CONFIG["total_virtual_epochs"]} x {CONFIG["steps_per_vepoch"]:,} steps = {total_steps:,} total steps')
print(f'Effective batch size: {CONFIG["batch_size"] * CONFIG["grad_accum_steps"]}')

In [ ]:
# Load data

base_path = '/kaggle/input/datasets/bangta205/train-val-test/hf_dataset-20260520T220937Z-3-002/hf_dataset'

from datasets import Dataset, DatasetDict, concatenate_datasets

try:
    train_part0 = Dataset.from_file(os.path.join(base_path, 'train', 'data-00000-of-00003.arrow'))
    train_part1 = Dataset.from_file(os.path.join(base_path, 'train', 'data-00001-of-00003.arrow'))
    train_part2 = Dataset.from_file(os.path.join(base_path, 'train', 'data-00002-of-00003.arrow'))

    train_dataset_full = concatenate_datasets([train_part0, train_part1, train_part2])

    dataset = DatasetDict({
        'train': train_dataset_full,
        'val':   Dataset.from_file(os.path.join(base_path, 'val', 'data-00000-of-00001.arrow')),
        'test':  Dataset.from_file(os.path.join(base_path, 'test', 'data-00000-of-00001.arrow'))
    })

    print(dataset)

    CONFIG['dataset_path'] = base_path

except Exception as e:
    print("Error structure:")
    print(e)

In [ ]:
# Tokenize (cached)
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

# Tinh log_max_subs
max_subs = 0
for batch in dataset['train'].iter(batch_size=100_000):
    m = max(batch['n_subscribers'])
    if m > max_subs: max_subs = m
log_max_subs = float(np.log1p(max_subs))
print(f'max_subs={max_subs:,}')

def preprocess(batch):
    texts = [str(t) if t else '' for t in batch['cleaned_text']]
    enc = tokenizer(texts, max_length=CONFIG['max_length'],
                    padding='max_length', truncation=True)
    enc['has_url']     = [float(v) for v in batch['has_url']]
    enc['verified']    = [float(v) for v in batch['verified']]
    enc['n_subs_norm'] = [float(np.log1p(v)) / log_max_subs for v in batch['n_subscribers']]
    enc['label']       = [int(v) for v in batch['label']]
    return enc

print('Tokenizing (cached sau lan dau)...')
tokenized = dataset.map(
    preprocess,
    batched=True,
    batch_size=10_000,
    num_proc=CONFIG['num_proc'],
    remove_columns=dataset['train'].column_names,
    desc='Tokenizing',
    keep_in_memory=True,
    load_from_cache_file=False
)

tokenized.set_format('torch', columns=['input_ids', 'attention_mask',
                                        'has_url', 'verified', 'n_subs_norm', 'label'])
print(tokenized)

In [ ]:
# persistent_workers + prefetch_factor

def collate_fn(batch):
    return {
        'input_ids'     : torch.stack([b['input_ids']      for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'metadata'      : torch.stack([
            torch.tensor([b['has_url'], b['verified'], b['n_subs_norm']], dtype=torch.float)
            for b in batch
        ]),
        'label': torch.stack([b['label'] for b in batch]),
    }

print('Computing sample weights...')
train_labels = np.array(tokenized['train']['label'])
class_counts = np.bincount(train_labels)
class_w      = 1.0 / class_counts
sample_w     = class_w[train_labels]
sampler      = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)
print(f'Class weights: not_scam={class_w[0]:.6f} | scam={class_w[1]:.2f}')

train_loader = DataLoader(
    tokenized['train'],
    batch_size=CONFIG['batch_size'],
    sampler=sampler,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)
val_loader = DataLoader(
    tokenized['val'],
    batch_size=CONFIG['batch_size'] * 4,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)
test_loader = DataLoader(
    tokenized['test'],
    batch_size=CONFIG['batch_size'] * 4,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True,
)

# Tao infinite iterator de khong bi het data giua virtual epoch
train_iter = cycle(train_loader)

print(f'DataLoaders ready')

In [ ]:
# Model – PhoBERT + LoRA
class ScamClassifier(nn.Module):
    def __init__(self, model_name, num_meta=3, num_classes=2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(hidden + num_meta, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask, metadata):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.head(torch.cat([cls, metadata], dim=1))


print('Building model...')
model = ScamClassifier(CONFIG['model_name'])

lora_cfg = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=CONFIG['lora_r'], lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    target_modules=['query', 'key', 'value'],
    bias='none'
)
model.encoder = get_peft_model(model.encoder, lora_cfg)
model.encoder.print_trainable_parameters()
model = model.to(device)

print(f'Model ready | LoRA r={CONFIG["lora_r"]}')

In [ ]:
# Focal Loss + Optimizer + Cosine Scheduler
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        ce  = nn.functional.cross_entropy(logits, targets, reduction='none')
        pt  = torch.exp(-ce)
        fl  = (1 - pt) ** self.gamma * ce
        a_t = torch.where(targets == 1,
                          torch.full_like(ce, self.alpha),
                          torch.full_like(ce, 1 - self.alpha))
        return (a_t * fl).mean()


criterion = FocalLoss(gamma=CONFIG['focal_gamma'], alpha=CONFIG['focal_alpha'])

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG['learning_rate'], weight_decay=0.01
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=CONFIG['warmup_steps'],
    num_training_steps=total_steps
)
scaler = torch.amp.GradScaler('cuda')

print(f'Optimizer ready | total_steps={total_steps:,}')

In [ ]:
# Checkpoint helpers

def save_checkpoint(vepoch, global_step, best_f1, history):
    torch.save({
        'vepoch'         : vepoch,
        'global_step'    : global_step,
        'model_state'    : model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state'   : scaler.state_dict(),
        'best_f1'        : best_f1,
    }, CKPT_PATH)
    with open(LOG_PATH, 'w') as f:
        json.dump(history, f, indent=2, ensure_ascii=False)
    print(f'  Checkpoint saved (vepoch={vepoch+1}, step={global_step:,}, best_f1={best_f1:.4f})')


def load_checkpoint():
    if not os.path.exists(CKPT_PATH):
        print('Khong co checkpoint, bat dau moi.')
        return 0, 0, 0.0, []
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    scaler.load_state_dict(ckpt['scaler_state'])
    history = json.load(open(LOG_PATH)) if os.path.exists(LOG_PATH) else []
    ve, gs, f = ckpt['vepoch'], ckpt['global_step'], ckpt['best_f1']
    print(f'Resume: vepoch={ve+1}, step={gs:,}, best_f1={f:.4f}')
    return ve, gs, f, history


print('Checkpoint helpers ready')

In [ ]:
# Evaluate

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, preds_all, labels_all = 0.0, [], []
    for batch in loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        meta = batch['metadata'].to(device)
        lbl  = batch['label'].to(device)
        with torch.amp.autocast('cuda'):
            logits = model(ids, mask, meta)
            loss   = criterion(logits, lbl)
        total_loss += loss.item()
        preds_all.extend(torch.argmax(logits, 1).cpu().tolist())
        labels_all.extend(lbl.cpu().tolist())
    model.train()
    return {
        'loss'     : total_loss / len(loader),
        'f1'       : f1_score(labels_all, preds_all, average='binary', zero_division=0),
        'precision': precision_score(labels_all, preds_all, average='binary', zero_division=0),
        'recall'   : recall_score(labels_all, preds_all, average='binary', zero_division=0),
        'accuracy' : sum(p==l for p,l in zip(preds_all, labels_all)) / len(labels_all),
    }

print('Evaluate helper ready')

In [ ]:
# STEP-BASED TRAINING LOOP
#
# Khong dung epoch. Thay vao do:
#   - train_iter = cycle(train_loader): lap lai vo han
#   - Moi virtual epoch = steps_per_vepoch steps (~1-1.5h)
#   - Eval + checkpoint sau moi virtual epoch
#   - Resume bat cu luc nao Colab ngat

start_vepoch, global_step, best_f1, history = load_checkpoint()
patience_counter = 0
N   = CONFIG['steps_per_vepoch']
GA  = CONFIG['grad_accum_steps']   # gradient accumulation

# Skip steps da lam trong iterator
if global_step > 0:
    print(f'Skipping {global_step % N} steps da lam trong virtual epoch nay...')
    for _ in range(global_step % N):
        next(train_iter)

print(f'\nTraining | Virtual epochs: {start_vepoch+1}->{CONFIG["total_virtual_epochs"]} | Steps/vepoch: {N:,}')
print(f'Grad accumulation: {GA} | Effective batch: {CONFIG["batch_size"]*GA}')

model.train()

for vepoch in range(start_vepoch, CONFIG['total_virtual_epochs']):
    vepoch_start = time.time()
    vepoch_loss  = 0.0

    optimizer.zero_grad()

    for step in range(N):
        batch = next(train_iter)

        ids  = batch['input_ids'].to(device, non_blocking=True)
        mask = batch['attention_mask'].to(device, non_blocking=True)
        meta = batch['metadata'].to(device, non_blocking=True)
        lbl  = batch['label'].to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            logits = model(ids, mask, meta)
            # Chia loss cho GA de gradient scale dung
            loss   = criterion(logits, lbl) / GA

        scaler.scale(loss).backward()
        vepoch_loss += loss.item() * GA

        # Chi update sau moi GA steps
        if (step + 1) % GA == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        global_step += 1

        # Log moi 1000 steps
        if (step + 1) % 1000 == 0:
            elapsed  = time.time() - vepoch_start
            avg_loss = vepoch_loss / (step + 1)
            steps_left = N - step - 1
            eta = steps_left * (elapsed / (step + 1))
            lr  = scheduler.get_last_lr()[0]
            print(f'  vE{vepoch+1} [{step+1}/{N}] loss={avg_loss:.4f} lr={lr:.2e} | {elapsed/60:.1f}min | ETA {eta/60:.1f}min')

    # --- Cuoi virtual epoch: Eval + Checkpoint ---
    elapsed = time.time() - vepoch_start
    m = evaluate(val_loader)
    history.append({
        'vepoch': vepoch + 1,
        'global_step': global_step,
        'train_loss': vepoch_loss / N,
        **{f'val_{k}': v for k, v in m.items()},
        'elapsed_min': elapsed / 60,
        'timestamp': datetime.now().isoformat()
    })

    print(f'\n=== Virtual Epoch {vepoch+1}/{CONFIG["total_virtual_epochs"]} ({elapsed/60:.1f}min) ===')
    print(f'  Train loss : {vepoch_loss/N:.4f}')
    print(f'  Val   loss : {m["loss"]:.4f}')
    print(f'  Val   F1   : {m["f1"]:.4f}  |  P={m["precision"]:.4f}  R={m["recall"]:.4f}')
    print(f'  Val   Acc  : {m["accuracy"]:.4f}')

    if m['f1'] > best_f1:
        best_f1 = m['f1']
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f'  NEW BEST F1={best_f1:.4f} -> saved best_model.pt')
        patience_counter = 0
    else:
        patience_counter += 1
        print(f'  No improvement ({patience_counter}/{CONFIG["early_stop_patience"]})')
        if patience_counter >= CONFIG['early_stop_patience']:
            print('Early stopping!')
            save_checkpoint(vepoch, global_step, best_f1, history)
            raise SystemExit

    save_checkpoint(vepoch, global_step, best_f1, history)
    print()

print(f'Training complete! Best Val F1 = {best_f1:.4f}')

In [ ]:
# Test Set Evaluation

print('Loading best model...')
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
test_m = evaluate(test_loader)
print('\n=== TEST SET RESULTS ===')
for k, v in test_m.items():
    print(f'  {k:12}: {v:.4f}')

In [ ]:
# Plot

import matplotlib.pyplot as plt

if history:
    vepochs = [h['vepoch']    for h in history]
    f1s     = [h['val_f1']    for h in history]
    losses  = [h['val_loss']  for h in history]
    times   = [h['elapsed_min'] for h in history]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(vepochs, f1s, marker='o')
    axes[0].set_title('Val F1 per Virtual Epoch'); axes[0].grid(True)
    axes[1].plot(vepochs, losses, marker='o', color='orange')
    axes[1].set_title('Val Loss per Virtual Epoch'); axes[1].grid(True)
    axes[2].plot(vepochs, times, marker='s', color='green')
    axes[2].set_title('Time per Virtual Epoch (min)'); axes[2].grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['output_dir'], 'training_curve.png'))
    plt.show()

In [ ]:
# Inference

# 1. Load weights
print("Loading weights...")
model.load_state_dict(torch.load('/kaggle/working/best_model.pt', map_location=device))
model.to(device)
model.eval()

# 2. Predict helper
def predict(texts):
    results = []
    with torch.no_grad():
        for text in texts:
            enc = tokenizer(text, max_length=CONFIG['max_length'], padding='max_length', truncation=True, return_tensors='pt')
            meta = torch.tensor([[len(text), text.count('?'), text.count('!')]], dtype=torch.float)
            with torch.amp.autocast('cuda'):
                probs = torch.softmax(model(enc['input_ids'].to(device), enc['attention_mask'].to(device), meta.to(device)), dim=1)
            p = probs[0]
            results.append(('SCAM' if p[1] > 0.5 else 'SAFE', round(p[1].item(), 4)))
    return results

# 3. Test samples
samples = [
    'Florentino',
    'Ta là chủ nhân mới của sức mạnh này',
    'Tôi đẹp trai',
    'Chúc mừng bạn đã trúng thưởng 100 triệu! Liên hệ ngay.',
    'Hôm nay đi chơi không mọi người?',
    'Đầu tư sinh lời 200%/tháng, hoàn vốn 100% bảo đảm.'
]

print("\nModel Predictions")
for text, (label, conf) in zip(samples, predict(samples)):
    print(f"[{label} | {conf:.3f}] {text}")